In [ ]:
!git clone https://github.com/d4-5/NLP4.git
%cd NLP4

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd()
print("Project root:", root)

processed_path = root / "data" / "processed_v2.csv"
gold_path = root / "data" / "sample" / "lab4_gold_ie.jsonl"
edge_path = root / "tests" / "ie_edge_cases.jsonl"

processed_df = pd.read_csv(processed_path)
print("processed_v2 shape:", processed_df.shape)
processed_df.head(3)

Project root: /home/user/lababd4
processed_v2 shape: (11161, 3)


,text_id,text_clean,sentences
0,text_0,"Дорогою зупинялися в селах, старцювали. Устимк...","['Дорогою зупинялися в селах, старцювали.', 'У..."
1,text_1,"То молитви продавав свої спудейські, то щось к...","['То молитви продавав свої спудейські, то щось..."
2,text_2,"Дуже часто дід досить точно вказував, куди сам...","['Дуже часто дід досить точно вказував, куди с..."


In [2]:
import json
from src.ie_rules import extract_dates, extract_amounts, extract_doc_ids, extract_all

currencies_path = root / "resources" / "currencies.json"
doc_ctx_path = root / "resources" / "doc_id_context_words.txt"

print("currencies.json exists:", currencies_path.exists())
print("doc_id_context_words.txt exists:", doc_ctx_path.exists())

with open(currencies_path, encoding="utf-8") as f:
    currencies = json.load(f)

print("Currencies loaded:", len(currencies))
print("Sample currencies:", list(currencies.items())[:5])


currencies.json exists: True
doc_id_context_words.txt exists: True
Currencies loaded: 13
Sample currencies: [('грн', 'UAH'), ('гривень', 'UAH'), ('гривня', 'UAH'), ('гривні', 'UAH'), ('₴', 'UAH')]


In [3]:
sample_texts = processed_df["text_clean"].dropna().sample(5, random_state=42).tolist()

for i, text in enumerate(sample_texts, 1):
    print(f"\n===== SAMPLE {i} =====")
    print(text[:220].replace("\n", " "))
    print(extract_all(text))



===== SAMPLE 1 =====
Ми також маємо спадок сотень поколінь богословів, отже, багато цитат отців Церкви увійшли в наші віросповідні книги.
{'DATE': [], 'AMOUNT': [], 'DOC_ID': []}

===== SAMPLE 2 =====
Тож коли Лянґе побачив, що все пов'язано, і все є Богом, він зрозумів, чому так лагідно, а водночас насмішкувато дивився на нього Штукенгайзен - адже Сам Господь нікуди не дівався від Адама, а лиш Адам відвертав своє обл
{'DATE': [], 'AMOUNT': [], 'DOC_ID': []}

===== SAMPLE 3 =====
Зазвичай патрулі ховаються десь поблизу таких місць і ловлять тих, хто не вміє читати табличок.
{'DATE': [], 'AMOUNT': [], 'DOC_ID': []}

===== SAMPLE 4 =====
Чоловік був у її владі, могла би навіть його убити, якби хотіла, могла не давати ні їсти, ні пити.
{'DATE': [], 'AMOUNT': [], 'DOC_ID': []}

===== SAMPLE 5 =====
Офіційно ТОВ "Юмвоса" не пов'язана з впливовими краматорчанами, але той факт що фірма багато років успішно конкурувала з бізнесом родини Близнюка каже про наявність впливових партнерів.
{'DATE'

In [4]:
from src.evaluate_ie import evaluate

metrics = evaluate(gold_path)
metrics


{'AMOUNT': {'correct_extractions': 10,
  'all_extractions': 12,
  'precision': 0.8333,
  'gold_size': 10},
 'DATE': {'correct_extractions': 12,
  'all_extractions': 19,
  'precision': 0.6316,
  'gold_size': 12},
 'DOC_ID': {'correct_extractions': 10,
  'all_extractions': 11,
  'precision': 0.9091,
  'gold_size': 10}}

In [5]:
import pandas as pd

rows = []
for field_type, vals in metrics.items():
    rows.append({
        "field_type": field_type,
        "correct_extractions": vals["correct_extractions"],
        "all_extractions": vals["all_extractions"],
        "precision": vals["precision"],
        "gold_size": vals["gold_size"],
    })

precision_df = pd.DataFrame(rows).sort_values("field_type").reset_index(drop=True)
precision_df


,field_type,correct_extractions,all_extractions,precision,gold_size
0,AMOUNT,10,12,0.8333,10
1,DATE,12,19,0.6316,12
2,DOC_ID,10,11,0.9091,10


In [6]:
import json

gold_rows = []
with open(gold_path, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            gold_rows.append(json.loads(line))

texts_by_id = {}
gold_keys = set()

def norm_gold_key(r):
    ft = r["field_type"]
    val = r["normalized_value"]
    if ft == "AMOUNT" and isinstance(val, dict):
        val = {
            "value": float(val.get("value")) if val.get("value") is not None else None,
            "currency": val.get("currency", "UNKNOWN"),
        }
    if ft == "DOC_ID" and isinstance(val, dict):
        val = {"type": val.get("type"), "value": val.get("value")}
    return (r["text_id"], ft, r["start_char"], r["end_char"], json.dumps(val, ensure_ascii=False, sort_keys=True))

for r in gold_rows:
    texts_by_id[r["text_id"]] = r["text"]
    gold_keys.add(norm_gold_key(r))

def pred_key(text_id, ent):
    ft = ent["field_type"]
    if ft == "AMOUNT":
        val = {"value": ent.get("value"), "currency": ent.get("currency", "UNKNOWN")}
    elif ft == "DOC_ID":
        val = {"type": ent.get("type"), "value": ent.get("value")}
    else:
        val = ent.get("value")
    return (text_id, ft, ent["start_char"], ent["end_char"], json.dumps(val, ensure_ascii=False, sort_keys=True))

all_preds = []
pred_keys = set()
for text_id, text in texts_by_id.items():
    ext = extract_all(text)
    for ft, ents in ext.items():
        for e in ents:
            k = pred_key(text_id, e)
            pred_keys.add(k)
            all_preds.append((text_id, text, e, k))

fp_keys = pred_keys - gold_keys
print("Total FP on gold subset:", len(fp_keys))

analysis_rows = []
for text_id, text, ent, key in all_preds:
    if key not in fp_keys:
        continue

    span = text[ent["start_char"]:ent["end_char"]]
    ft = ent["field_type"]

    if ft == "DATE":
        reason = "Дата витягнута коректно regex, але відсутня у поточній ручній розмітці gold (incomplete gold)."
        anti_rule = "Під час розмітки: маркувати всі валідні DATE у вибраному тексті (повна розмітка)."
    elif ft == "AMOUNT":
        reason = "Сума витягнута в шумному/неоднозначному контексті або не потрапила у gold."
        anti_rule = "Додати контекстне anti-rule: відкидати AMOUNT поруч із технічними патернами/без семантики платежу."
    else:
        reason = "№-патерн спрацював у слабкому контексті."
        anti_rule = "Підсилити DOC_ID anti-rule: жорсткіший словник блокліста + обов'язковий контекстний якір."

    analysis_rows.append({
        "text_id": text_id,
        "field_type": ft,
        "span": span,
        "why_fp": reason,
        "added_anti_rule": anti_rule,
        "context": text[max(0, ent["start_char"]-45):ent["end_char"]+45],
    })

analysis_df = pd.DataFrame(analysis_rows).head(10)
analysis_df


Total FP on gold subset: 10


,text_id,field_type,span,why_fp,added_anti_rule,context
0,text_9326,DATE,19.02.2018,"Дата витягнута коректно regex, але відсутня у ...",Під час розмітки: маркувати всі валідні DATE у...,"від 14.02.2018, лист народного депутата від 1..."
1,text_9326,DATE,21.03.2018,"Дата витягнута коректно regex, але відсутня у ...",Під час розмітки: маркувати всі валідні DATE у...,го ВП ГУ Нацполіції в Луганській області від 2...
2,text_9293,AMOUNT,3460 грн,Сума витягнута в шумному/неоднозначному контек...,Додати контекстне anti-rule: відкидати AMOUNT ...,"артири, 8 двокімнатних та 22 трикімнатних по 3..."
3,text_9293,AMOUNT,4180 грн,Сума витягнута в шумному/неоднозначному контек...,Додати контекстне anti-rule: відкидати AMOUNT ...,"е житла в житомирських новобудовах, а саме - 4..."
4,text_9843,DATE,26.04.2016,"Дата витягнута коректно regex, але відсутня у ...",Під час розмітки: маркувати всі валідні DATE у...,мінальне провадження № 42016050000000308 від 2...
5,text_10054,DATE,03.06.2014,"Дата витягнута коректно regex, але відсутня у ...",Під час розмітки: маркувати всі валідні DATE у...,сним рішення АМКУ у спрві №136-26.386-12 від 0...
6,text_10054,DOC_ID,№321-p,№-патерн спрацював у слабкому контексті.,Підсилити DOC_ID anti-rule: жорсткіший словник...,я АМКУ у спрві №136-26.386-12 від 03.06.2014 №...
7,text_10224,DATE,04.01.2017,"Дата витягнута коректно regex, але відсутня у ...",Під час розмітки: маркувати всі валідні DATE у...,льного провадження у № 42017171090000003 від 0...
8,text_10345,DATE,08.10.2016,"Дата витягнута коректно regex, але відсутня у ...",Під час розмітки: маркувати всі валідні DATE у...,мінальне провадження № 42016000000002788 від 0...
9,text_10391,DATE,24.02.2017,"Дата витягнута коректно regex, але відсутня у ...",Під час розмітки: маркувати всі валідні DATE у...,нального провадження № 42017221050000013 від 2...
